# Notebook 05 — Product & Geographic Intelligence

**Project:** E-Commerce Customer Intelligence & Sales Analytics  
**Programme:** IBM SkillsBuild Data Analytics with AI Internship 2026  
**Dataset:** Online Retail II (`online_retail_II.xlsx`)  

---

## Business Objective

This notebook answers:

1. Which products drive the most revenue, volume, and customer reach?
2. How concentrated is product revenue across the catalogue?
3. Which products have high return rates — and why?
4. Which countries generate the most revenue?
5. How do UK vs international customer economics compare?
6. Which products are geographically concentrated?

**Prerequisite notebooks:** 01 → 04 must be executed first.  
**Runner script:** `src/nb05_runner.py` — execute before this notebook.  
**Outputs:** `outputs/product_summary.csv`, `outputs/country_summary.csv`, figures 31–42 in `outputs/figures/`.

---

## Pipeline Reminder

All analysis uses the shared cleaned datasets produced by NB02/NB03:

| Dataset | Description |
|---------|-------------|
| `vmt` | Valid merchandise transactions (positive revenue, genuine products) — 1,016,911 rows |
| `mpt` | Merchandise + product-level cancellations (positive + negative, no service codes) — 1,040,855 rows |
| `customer_transactions` | Identified-customer transactions only — 784,513 rows |

---
## 1. Imports & Setup

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')

DATA_DIR    = Path('../data')
OUTPUT_DIR  = Path('../outputs')
FIGURES_DIR = OUTPUT_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded.')
print(f'Figures will be saved to: {FIGURES_DIR.resolve()}')

---
## 2. Load Cleaned Data

Re-build the shared pipeline (dedup → classify → assign transaction type → canonical description).  
This mirrors NB02/NB03 exactly — no data is modified.

In [ ]:
# ── Load both sheets ──────────────────────────────────────────────────────────
xl = pd.ExcelFile(DATA_DIR / 'online_retail_II.xlsx', engine='openpyxl')
raw = pd.concat(
    [pd.read_excel(xl, sheet_name=s, dtype={'Customer ID': str})
     for s in xl.sheet_names],
    ignore_index=True
)
print(f'Raw rows loaded : {len(raw):,}')

# ── Exact-duplicate removal ───────────────────────────────────────────────────
df = raw.drop_duplicates()
print(f'After dedup     : {len(df):,}  ({len(raw)-len(df):,} exact duplicates removed)')

# ── Parse dates ───────────────────────────────────────────────────────────────
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# ── Revenue column ────────────────────────────────────────────────────────────
df['revenue'] = df['Quantity'] * df['Price']

# ── StockCode classifier ──────────────────────────────────────────────────────
NUMERIC_SC   = df['StockCode'].astype(str).str.match(r'^\d{5}[A-Za-z]{0,2}$')
SERVICE_CODES = {'POST', 'D', 'M', 'BANK CHARGES', 'PADS', 'DOT',
                 'CRUK', 'C2', 'AMAZONFEE', 'S', 'ADJUST', 'ADJUST2',
                 'B', 'GIFT_0001_10', 'GIFT_0001_20', 'GIFT_0001_30',
                 'GIFT_0001_40', 'GIFT_0001_50', 'TEST001', 'TEST002', 'm'}
svc_mask = df['StockCode'].astype(str).str.upper().isin(
    [s.upper() for s in SERVICE_CODES]
)
df['sc_type'] = np.where(NUMERIC_SC, 'merchandise',
                np.where(svc_mask, 'service', 'other'))

# ── Transaction type ──────────────────────────────────────────────────────────
c_inv_mask = df['Invoice'].astype(str).str.startswith('C')
def assign_tx_type(row):
    if row['sc_type'] == 'service':       return 'SERVICE'
    if c_inv_mask[row.name]:              return 'CANCELLED_INVOICE'
    if row['Quantity'] < 0:               return 'NEGATIVE_QTY'
    if row['Price'] <= 0:                 return 'ZERO_PRICE'
    return 'SALE'

df['transaction_type'] = df.apply(assign_tx_type, axis=1)

# ── Canonical description ─────────────────────────────────────────────────────
modal_desc = (
    df[df['sc_type'] == 'merchandise']
    .groupby('StockCode')['Description']
    .agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0])
)
df['canonical_description'] = df['StockCode'].map(modal_desc).fillna(df['Description'])

# ── Analytical subsets ────────────────────────────────────────────────────────
vmt = df[(df['transaction_type'] == 'SALE') & (df['sc_type'] == 'merchandise')].copy()
mpt = df[df['sc_type'] == 'merchandise'].copy()
customer_transactions = vmt[vmt['Customer ID'].notna()].copy()

print(f'vmt rows        : {len(vmt):,}')
print(f'mpt rows        : {len(mpt):,}')
print(f'customer_txns   : {len(customer_transactions):,}')

---
## 3. Product Analytical Table

Build one row per StockCode summarising gross revenue, net revenue, returns, units, orders, and customers.

In [ ]:
# ── Positive (SALE) aggregations ──────────────────────────────────────────────
pos = (
    vmt.groupby('StockCode').agg(
        canonical_description=('canonical_description', 'first'),
        gross_revenue        =('revenue',   'sum'),
        positive_units       =('Quantity',  'sum'),
        orders               =('Invoice',   'nunique'),
        customers            =('Customer ID', lambda x: x.dropna().nunique()),
        avg_price_list       =('Price',     'mean'),
    ).reset_index()
)

# Average selling price weighted by quantity
wasp = (
    vmt.groupby('StockCode')
    .apply(lambda g: (g['revenue'].sum() / g['Quantity'].sum()) if g['Quantity'].sum() > 0 else 0)
    .rename('average_selling_price').reset_index()
)
pos = pos.merge(wasp, on='StockCode', how='left')

# ── Negative (CANCELLED_INVOICE) aggregations ─────────────────────────────────
neg = (
    mpt[mpt['transaction_type'] == 'CANCELLED_INVOICE']
    .groupby('StockCode').agg(
        return_value   =('revenue',  'sum'),
        negative_units =('Quantity', 'sum'),
    ).reset_index()
)

# ── Combine ───────────────────────────────────────────────────────────────────
prod_tbl = pos.merge(neg, on='StockCode', how='left')
prod_tbl['return_value']   = prod_tbl['return_value'].fillna(0)
prod_tbl['negative_units'] = prod_tbl['negative_units'].fillna(0)
prod_tbl['net_revenue']    = prod_tbl['gross_revenue'] + prod_tbl['return_value']
prod_tbl['return_rate_pct'] = np.where(
    prod_tbl['gross_revenue'] > 0,
    abs(prod_tbl['return_value']) / prod_tbl['gross_revenue'] * 100,
    np.nan
)
prod_tbl['revenue_per_customer'] = np.where(
    prod_tbl['customers'] > 0,
    prod_tbl['gross_revenue'] / prod_tbl['customers'],
    np.nan
)

# Active products = positive gross revenue
prod_active = prod_tbl[prod_tbl['gross_revenue'] > 0].copy()
n_active = len(prod_active)

print(f'Total products in prod_tbl : {len(prod_tbl):,}')
print(f'Active products (gross > 0): {n_active:,}')
print(f'Gross revenue reconciles   : £{prod_active["gross_revenue"].sum():,.2f}')
print(f'Net revenue reconciles     : £{prod_tbl["net_revenue"].sum():,.2f}')

---
## 4. Product Performance Rankings

In [ ]:
RANK_COLS_NET  = ['StockCode', 'canonical_description', 'gross_revenue', 'net_revenue',
                  'positive_units', 'orders', 'customers']
RANK_COLS_UNIT = ['StockCode', 'canonical_description', 'positive_units',
                  'gross_revenue', 'orders', 'customers']
RANK_COLS_CUST = ['StockCode', 'canonical_description', 'customers',
                  'gross_revenue', 'positive_units']

pa = prod_active.sort_values('net_revenue', ascending=False)

print('Top 15 by NET REVENUE:')
display(pa[RANK_COLS_NET].head(15).set_index('StockCode'))

print('\nTop 15 by UNITS SOLD:')
display(prod_active.sort_values('positive_units', ascending=False)[RANK_COLS_UNIT].head(15).set_index('StockCode'))

print('\nTop 15 by CUSTOMER REACH:')
display(prod_active.sort_values('customers', ascending=False)[RANK_COLS_CUST].head(15).set_index('StockCode'))

---
## 5. Product Revenue Concentration (Pareto Analysis)

How many products are needed to reach 25%, 50%, 75%, 80%, and 90% of net revenue?

In [ ]:
pa_sorted = prod_active.sort_values('net_revenue', ascending=False).copy()
pa_sorted['cumrev']      = pa_sorted['net_revenue'].cumsum()
pa_sorted['cum_rev_pct'] = pa_sorted['cumrev'] / pa_sorted['net_revenue'].sum() * 100

def n_for_pct(target):
    return int((pa_sorted['cum_rev_pct'] <= target).sum()) + 1

thresholds = [25, 50, 75, 80, 90]
results = []
for t in thresholds:
    n = n_for_pct(t)
    p = n / n_active * 100
    results.append({'Threshold': f'{t}% of net revenue', 'Products needed': n,
                    '% of catalogue': f'{p:.1f}%'})

conc_df = pd.DataFrame(results)
display(conc_df)

n25, n50, n75, n80, n90 = [n_for_pct(t) for t in thresholds]
p25 = n25/n_active*100; p50 = n50/n_active*100
p75 = n75/n_active*100; p80 = n80/n_active*100; p90 = n90/n_active*100

print(f'\nKey finding: {n50} products ({p50:.1f}% of catalogue) account for 50% of net revenue.')
print(f'Key finding: {n80} products ({p80:.1f}% of catalogue) account for 80% of net revenue.')

---
## 6. Product Volume vs Value Analysis

In [ ]:
corr_uv = prod_active['positive_units'].corr(prod_active['net_revenue'])
corr_pv = prod_active['average_selling_price'].corr(prod_active['net_revenue'])

med_units = prod_active['positive_units'].median()
med_price = prod_active['average_selling_price'].median()

high_vol  = prod_active['positive_units']       > med_units
high_price= prod_active['average_selling_price'] > med_price

hv_hq = prod_active[ high_vol &  high_price]
hp_lv = prod_active[~high_vol &  high_price]
hv_lp = prod_active[ high_vol & ~high_price]
lv_lp = prod_active[~high_vol & ~high_price]

print(f'Correlation: units vs net revenue   = {corr_uv:.3f}')
print(f'Correlation: avg price vs net rev   = {corr_pv:.3f}')
print(f'Median units per product            = {med_units:,.0f}')
print(f'Median avg selling price            = £{med_price:.2f}')
print()
print(f'High-volume + high-price (stars)    : {len(hv_hq):,} products')
print(f'High-price + low-volume (niche)     : {len(hp_lv):,} products')
print(f'High-volume + low-price (commodities): {len(hv_lp):,} products')
print(f'Low-volume + low-price (long tail)  : {len(lv_lp):,} products')

---
## 7. Product Return Rate Analysis

Only products with ≥ 10 orders are included to avoid noise from single-transaction products.

In [ ]:
pa_ret = prod_active[prod_active['orders'] >= 10].copy()

print(f'Products with >= 10 orders: {len(pa_ret):,}')
print('\nReturn rate distribution (% of gross revenue returned):')
display(pa_ret['return_rate_pct'].describe(percentiles=[.25, .5, .75, .90, .95, .99]))

print('\nTop 15 by VALUE return rate:')
display(
    pa_ret.sort_values('return_rate_pct', ascending=False)
    [['StockCode','canonical_description','gross_revenue','return_value',
      'net_revenue','return_rate_pct','orders','positive_units']].head(15)
    .set_index('StockCode')
)

---
## 8. Deep Dive: StockCode 23166 — MEDIUM CERAMIC TOP STORAGE JAR

This product appears in the top-10 by gross revenue but has a 94.8% value return rate.  
Investigation determines whether this reflects a genuine problem or a data artefact.

In [ ]:
rows_23166 = mpt[mpt['StockCode'] == '23166'].copy()
pos_23166  = rows_23166[rows_23166['revenue'] > 0]
neg_23166  = rows_23166[rows_23166['revenue'] < 0]

print(f'Total rows in mpt for 23166 : {len(rows_23166)}')
print(f'Unique descriptions         : {rows_23166["Description"].dropna().unique().tolist()}')
print()
print('Transaction type breakdown:')
print(rows_23166['transaction_type'].value_counts())
print()
print(f'Positive rows: {len(pos_23166)} | Gross revenue: £{pos_23166["revenue"].sum():,.2f}')
print(f'Negative rows: {len(neg_23166)} | Return value : £{neg_23166["revenue"].sum():,.2f}')
print(f'Net revenue  : £{rows_23166["revenue"].sum():,.2f}')

# The key transaction
print('\n--- The bulk order + same-day cancellation ---')
key_inv = rows_23166[rows_23166['Invoice'].astype(str).isin(['541431', 'C541433'])]
display(key_inv[['Invoice','InvoiceDate','Quantity','Price','revenue','Customer ID','Country']])

print('\nInvoice prefixes on negative rows:')
print(neg_23166['Invoice'].astype(str).str[0].value_counts())

In [ ]:
print('--- Investigation conclusion for 23166 ---')
print()
print('Finding: All 10 negative rows carry C-prefix invoice numbers.')
print('The dominant event is Invoice 541431 (74,215 units, £77,183.60)')
print('cancelled by C541433 on the same day (2011-01-18).')
print('This single bulk wholesale order+cancellation explains 99.6% of the return value.')
print('Remaining transactions (May–Dec 2011) are normal small retail orders (1–288 units).')
print()
print('Decision: Retain in analysis. The high return rate metric is driven by a single')
print('one-off wholesale event, not a persistent returns problem.')
print(f'Excluding the bulk pair, net revenue = £{rows_23166[~rows_23166["Invoice"].astype(str).isin(["541431","C541433"])]["revenue"].sum():,.2f}')

---
## 9. Deep Dive: StockCode 20879 — TREE OF NOAH FESTIVE SCENTED CANDLE

Returns exceed gross sales (297% return rate). Investigation determines the cause.

In [ ]:
rows_20879 = mpt[mpt['StockCode'] == '20879'].copy()
pos_20879  = rows_20879[rows_20879['revenue'] > 0]
neg_20879  = rows_20879[rows_20879['revenue'] < 0]
c_inv_20879  = rows_20879[rows_20879['Invoice'].astype(str).str.startswith('C')]
noc_inv_20879= rows_20879[
    ~rows_20879['Invoice'].astype(str).str.startswith('C') &
     (rows_20879['revenue'] < 0)
]

print(f'Total rows in mpt for 20879 : {len(rows_20879)}')
print(f'Positive rows : {len(pos_20879)} | Gross: £{pos_20879["revenue"].sum():.2f}')
print(f'Negative rows : {len(neg_20879)} | Return: £{neg_20879["revenue"].sum():.2f}')
print(f'Net revenue   : £{rows_20879["revenue"].sum():.2f}')
print()
print('All transactions:')
display(rows_20879[['Invoice','InvoiceDate','Quantity','Price','revenue',
                     'transaction_type','Customer ID','Country']].sort_values('InvoiceDate'))

print('\n--- Investigation conclusion for 20879 ---')
print('Returns exceed gross sales because cancellation invoices reference')
print('transactions that likely pre-date the dataset window, or the return')
print('quantities/prices differ from the in-window sales.')
print('All negative rows carry C-prefix invoices — properly recorded cancellations.')
print('This is a data-window artefact, not a real economic anomaly.')
print('Decision: Retain in mpt; net_revenue will be negative for this product.')
print('Exclude from net-revenue rankings but note in return-rate audit.')

---
## 10. Country Analysis

Build one row per country summarising revenue, orders, customers, AOV, and return rate.

In [ ]:
country_pos = (
    vmt.groupby('Country').agg(
        gross_revenue       =('revenue',    'sum'),
        orders              =('Invoice',    'nunique'),
        customers           =('Customer ID', lambda x: x.dropna().nunique()),
        positive_units      =('Quantity',   'sum'),
    ).reset_index()
)
country_neg = (
    mpt[mpt['transaction_type'] == 'CANCELLED_INVOICE']
    .groupby('Country').agg(return_value=('revenue', 'sum')).reset_index()
)
country_tbl = country_pos.merge(country_neg, on='Country', how='left')
country_tbl['return_value'] = country_tbl['return_value'].fillna(0)
country_tbl['net_revenue']  = country_tbl['gross_revenue'] + country_tbl['return_value']
country_tbl['gross_aov']    = country_tbl['gross_revenue'] / country_tbl['orders']
country_tbl['net_aov']      = country_tbl['net_revenue']   / country_tbl['orders']
country_tbl['revenue_per_customer'] = np.where(
    country_tbl['customers'] > 0,
    country_tbl['gross_revenue'] / country_tbl['customers'], np.nan
)
country_tbl['avg_units_per_order'] = country_tbl['positive_units'] / country_tbl['orders']
country_tbl['return_rate_pct'] = np.where(
    country_tbl['gross_revenue'] > 0,
    abs(country_tbl['return_value']) / country_tbl['gross_revenue'] * 100, np.nan
)
total_net = country_tbl['net_revenue'].sum()
country_tbl['net_rev_share'] = country_tbl['net_revenue'] / total_net * 100
country_tbl = country_tbl.sort_values('net_revenue', ascending=False).reset_index(drop=True)

print(f'Countries: {len(country_tbl)}')
display(
    country_tbl[['Country','gross_revenue','return_value','net_revenue',
                 'orders','customers','gross_aov','net_aov',
                 'revenue_per_customer','avg_units_per_order','return_rate_pct']]
    .head(20).set_index('Country')
)

---
## 11. UK vs International Split

In [ ]:
country_tbl['geo_group'] = np.where(country_tbl['Country'] == 'United Kingdom', 'UK', 'International')
geo = (
    country_tbl.groupby('geo_group').agg(
        n_countries          =('Country',              'count'),
        gross_revenue        =('gross_revenue',        'sum'),
        net_revenue          =('net_revenue',          'sum'),
        orders               =('orders',               'sum'),
        customers            =('customers',            'sum'),
        return_value         =('return_value',         'sum'),
    ).reset_index()
)
geo['gross_aov'] = geo['gross_revenue'] / geo['orders']
geo['net_aov']   = geo['net_revenue']   / geo['orders']
geo['revenue_per_customer'] = geo['gross_revenue'] / geo['customers']
geo['return_rate_pct'] = abs(geo['return_value']) / geo['gross_revenue'] * 100
geo['net_rev_share'] = geo['net_revenue'] / geo['net_revenue'].sum() * 100

display(geo.set_index('geo_group'))

uk_net  = geo.loc[geo['geo_group']=='UK','net_revenue'].values[0]
int_net = geo.loc[geo['geo_group']=='International','net_revenue'].values[0]
uk_rpc  = geo.loc[geo['geo_group']=='UK','revenue_per_customer'].values[0]
int_rpc = geo.loc[geo['geo_group']=='International','revenue_per_customer'].values[0]
uk_aov  = geo.loc[geo['geo_group']=='UK','gross_aov'].values[0]
int_aov = geo.loc[geo['geo_group']=='International','gross_aov'].values[0]

print(f'\nUK net revenue           : £{uk_net:,.2f} ({uk_net/total_net*100:.1f}%)')
print(f'International net revenue: £{int_net:,.2f} ({int_net/total_net*100:.1f}%)')
print(f'UK revenue per customer  : £{uk_rpc:,.2f}')
print(f'Intl revenue per customer: £{int_rpc:,.2f}')
print(f'UK gross AOV             : £{uk_aov:,.2f}')
print(f'International gross AOV  : £{int_aov:,.2f}')

---
## 12. Geographic Concentration

In [ ]:
ct_sorted = country_tbl.sort_values('net_revenue', ascending=False).reset_index(drop=True)
ct_sorted['cum_net_pct'] = ct_sorted['net_revenue'].cumsum() / total_net * 100

def s_top_n(n):
    return ct_sorted.head(n)['net_revenue'].sum() / total_net * 100

s1c, s3c, s5c, s10c = s_top_n(1), s_top_n(3), s_top_n(5), s_top_n(10)

def ctries_for_pct(t):
    return int((ct_sorted['cum_net_pct'] <= t).sum()) + 1

c50g, c75g, c80g, c90g = ctries_for_pct(50), ctries_for_pct(75), ctries_for_pct(80), ctries_for_pct(90)

print('Geographic concentration:')
for n, s in [(1,s1c),(3,s3c),(5,s5c),(10,s10c)]:
    print(f'  Top {n:2d} countr{"y" if n==1 else "ies"}: {s:.1f}% of net revenue')
print()
for t, c in [(50,c50g),(75,c75g),(80,c80g),(90,c90g)]:
    print(f'  Countries needed for {t}% net revenue: {c}')

---
## 13. Country Economics (Markets with ≥ 10 customers & ≥ 20 orders)

In [ ]:
country_eco = country_tbl[
    (country_tbl['customers'] >= 10) & (country_tbl['orders'] >= 20)
].copy().sort_values('gross_aov', ascending=False)

print(f'Countries with >= 10 customers and >= 20 orders: {len(country_eco)}')
display(
    country_eco[['Country','gross_aov','net_aov','revenue_per_customer',
                 'avg_units_per_order','orders','customers']]
    .set_index('Country')
)

---
## 14. Product × Country Analysis

In [ ]:
prod_country = (
    vmt.groupby(['StockCode','canonical_description','Country'])
    .agg(
        gross_revenue=('revenue',  'sum'),
        orders       =('Invoice',  'nunique'),
        units        =('Quantity', 'sum'),
    ).reset_index()
)

neg_pc = (
    mpt[mpt['transaction_type']=='CANCELLED_INVOICE']
    .groupby(['StockCode','Country'])
    .agg(return_value=('revenue','sum')).reset_index()
)
prod_country = prod_country.merge(neg_pc, on=['StockCode','Country'], how='left')
prod_country['return_value'] = prod_country['return_value'].fillna(0)
prod_country['net_revenue']  = prod_country['gross_revenue'] + prod_country['return_value']

# Products concentrated in one geography
prod_by_country_gross = (
    prod_country.groupby('StockCode')['gross_revenue'].transform('sum')
)
prod_country['country_gross_share'] = prod_country['gross_revenue'] / prod_by_country_gross * 100

min_gross = 5000
prod_gross_total = prod_country.groupby('StockCode')['gross_revenue'].sum()
eligible_skus = prod_gross_total[prod_gross_total >= min_gross].index

prod_conc = prod_country[
    (prod_country['country_gross_share'] >= 80) &
    (prod_country['StockCode'].isin(eligible_skus))
]
prod_conc_filtered = prod_conc[prod_conc['Country'] != 'United Kingdom']

# UK-exclusive products (>= 80% from UK, min £5k)
uk_conc_prods = prod_conc[prod_conc['Country'] == 'United Kingdom']

print(f'Products >= £5k gross where one country accounts for >= 80% of revenue : {len(prod_conc)}')
print(f'  Of which non-UK dominated : {len(prod_conc_filtered)}')
print(f'  Of which UK-dominated     : {len(uk_conc_prods)}')

---
## 15. Product-Market Opportunity Framework

Classify each product into quadrants based on return rate and revenue concentration.

In [ ]:
pa_opp = pa_ret.copy()
rev_median   = pa_opp['net_revenue'].median()
ret_threshold = 10.0  # > 10% return rate = high returns

pa_opp['opp_quad'] = np.where(
    (pa_opp['net_revenue'] >= rev_median) & (pa_opp['return_rate_pct'] <= ret_threshold),
    'Core (High Rev, Low Returns)',
    np.where(
        (pa_opp['net_revenue'] >= rev_median) & (pa_opp['return_rate_pct'] > ret_threshold),
        'Review (High Rev, High Returns)',
        np.where(
            (pa_opp['net_revenue'] < rev_median) & (pa_opp['return_rate_pct'] <= ret_threshold),
            'Growth (Low Rev, Low Returns)',
            'Rationalise (Low Rev, High Returns)'
        )
    )
)

opp_summary = pa_opp.groupby('opp_quad').agg(
    products   =('StockCode',   'count'),
    net_revenue=('net_revenue', 'sum'),
).reset_index()
opp_summary['rev_share'] = opp_summary['net_revenue'] / opp_summary['net_revenue'].sum() * 100

print('Product-market opportunity framework (products with >= 10 orders):')
display(opp_summary.set_index('opp_quad'))

---
## 16. Visualizations

Figures are saved to `outputs/figures/` with sequential numbering (31–42).

In [ ]:
def save_fig(name):
    path = FIGURES_DIR / name
    plt.savefig(path, dpi=150, bbox_inches='tight')
    print(f'Saved: {path}')
    plt.show()

In [ ]:
# Fig 31 — Top 20 products by net revenue
top20 = pa.head(20).sort_values('net_revenue')
fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(top20['canonical_description'], top20['net_revenue']/1000,
               color='steelblue', edgecolor='white', linewidth=0.4)
ax.set_xlabel('Net Revenue (£000s)')
ax.set_title('Top 20 Products by Net Revenue', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}k'))
plt.tight_layout()
save_fig('31_top20_products_net_revenue.png')

In [ ]:
# Fig 32 — Pareto curve: cumulative revenue share
fig, ax = plt.subplots(figsize=(9, 5))
x = range(1, len(pa_sorted)+1)
ax.plot(x, pa_sorted['cum_rev_pct'], color='steelblue', lw=1.5)
for t, n in [(50,n50),(80,n80),(90,n90)]:
    ax.axhline(t, color='crimson', ls='--', lw=0.8, alpha=0.7)
    ax.axvline(n, color='crimson', ls='--', lw=0.8, alpha=0.7)
    ax.annotate(f'{n} products → {t}%', xy=(n, t),
                xytext=(n+120, t-8), fontsize=8, color='crimson')
ax.set_xlabel('Number of Products (sorted by net revenue)')
ax.set_ylabel('Cumulative Net Revenue Share (%)')
ax.set_title('Product Revenue Concentration (Pareto Curve)', fontsize=13, fontweight='bold')
ax.set_xlim(0, n_active)
ax.set_ylim(0, 101)
plt.tight_layout()
save_fig('32_product_pareto_curve.png')

In [ ]:
# Fig 33 — Product quadrant scatter (volume vs price)
fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(
    prod_active['positive_units'], prod_active['average_selling_price'],
    c=np.log1p(prod_active['net_revenue']), cmap='YlOrRd',
    s=20, alpha=0.6, edgecolors='none'
)
ax.axvline(med_units, color='grey', ls='--', lw=0.8, label=f'Median units ({med_units:,.0f})')
ax.axhline(med_price, color='navy', ls='--', lw=0.8, label=f'Median price (£{med_price:.2f})')
plt.colorbar(sc, ax=ax, label='log(Net Revenue)')
ax.set_xlabel('Positive Units Sold')
ax.set_ylabel('Average Selling Price (£)')
ax.set_title('Product Quadrant: Volume vs Price\n(colour = log net revenue)',
             fontsize=12, fontweight='bold')
ax.set_xscale('log'); ax.set_yscale('log')
ax.legend(fontsize=9)
plt.tight_layout()
save_fig('33_product_volume_price_quadrant.png')

In [ ]:
# Fig 34 — Return rate distribution (< 50% cap for readability)
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(pa_ret['return_rate_pct'].clip(upper=50), bins=50,
        color='coral', edgecolor='white', linewidth=0.4)
ax.axvline(pa_ret['return_rate_pct'].median(), color='navy', ls='--', lw=1.2,
           label=f'Median {pa_ret["return_rate_pct"].median():.1f}%')
ax.axvline(10, color='crimson', ls=':', lw=1.2, label='10% threshold')
ax.set_xlabel('Return Rate % (capped at 50%)')
ax.set_ylabel('Number of Products')
ax.set_title('Product Return Rate Distribution\n(products with >= 10 orders)',
             fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
save_fig('34_product_return_rate_distribution.png')

In [ ]:
# Fig 35 — Top 15 products by return rate (value, >= 10 orders)
top_ret = pa_ret.sort_values('return_rate_pct', ascending=False).head(15)
top_ret_sorted = top_ret.sort_values('return_rate_pct')
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top_ret_sorted['canonical_description'], top_ret_sorted['return_rate_pct'],
        color='coral', edgecolor='white', linewidth=0.4)
ax.set_xlabel('Return Rate (% of Gross Revenue)')
ax.set_title('Top 15 Products by Value Return Rate\n(min 10 orders)',
             fontsize=12, fontweight='bold')
ax.axvline(10, color='crimson', ls='--', lw=1, label='10% threshold')
ax.legend()
plt.tight_layout()
save_fig('35_top15_high_return_products.png')

In [ ]:
# Fig 36 — 23166 timeline: monthly net revenue excluding bulk event
rows_23166_excl = rows_23166[
    ~rows_23166['Invoice'].astype(str).isin(['541431','C541433'])
].copy()
rows_23166_excl['month'] = rows_23166_excl['InvoiceDate'].dt.to_period('M')
monthly_23166 = rows_23166_excl.groupby('month')['revenue'].sum().reset_index()
monthly_23166['month_str'] = monthly_23166['month'].astype(str)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(monthly_23166['month_str'], monthly_23166['revenue'],
       color='steelblue', edgecolor='white')
ax.set_xlabel('Month')
ax.set_ylabel('Net Revenue (£)')
ax.set_title('StockCode 23166 — Monthly Net Revenue (bulk pair excluded)',
             fontsize=11, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
save_fig('36_23166_monthly_revenue.png')

In [ ]:
# Fig 37 — 20879 all transactions
fig, ax = plt.subplots(figsize=(9, 4))
colors = rows_20879['revenue'].apply(lambda v: 'steelblue' if v > 0 else 'coral')
ax.bar(range(len(rows_20879)), rows_20879['revenue'], color=colors, edgecolor='white')
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('Transaction Index')
ax.set_ylabel('Revenue (£)')
ax.set_title('StockCode 20879 — All Transactions (blue=sale, red=return)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
save_fig('37_20879_all_transactions.png')

In [ ]:
# Fig 38 — Top 15 countries by net revenue
top15c = country_tbl.head(15).sort_values('net_revenue')
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top15c['Country'], top15c['net_revenue']/1000,
        color='steelblue', edgecolor='white', linewidth=0.4)
ax.set_xlabel('Net Revenue (£000s)')
ax.set_title('Top 15 Countries by Net Revenue', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}k'))
plt.tight_layout()
save_fig('38_top15_countries_net_revenue.png')

In [ ]:
# Fig 39 — UK vs International: key metrics comparison
metrics = ['gross_aov', 'revenue_per_customer', 'return_rate_pct']
labels  = ['Gross AOV (£)', 'Revenue per Customer (£)', 'Return Rate (%)']

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
for ax, m, lbl in zip(axes, metrics, labels):
    vals  = [geo.loc[geo['geo_group']=='UK',     m].values[0],
             geo.loc[geo['geo_group']=='International', m].values[0]]
    ax.bar(['UK','International'], vals,
           color=['steelblue','coral'], edgecolor='white')
    ax.set_title(lbl, fontsize=10, fontweight='bold')
    ax.set_ylabel(lbl)
    for i, v in enumerate(vals):
        ax.text(i, v*1.02, f'{v:,.1f}', ha='center', va='bottom', fontsize=9)
fig.suptitle('UK vs International: Key Economics', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('39_uk_vs_international_economics.png')

In [ ]:
# Fig 40 — Geographic concentration curve
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, len(ct_sorted)+1), ct_sorted['cum_net_pct'],
        color='steelblue', lw=2, marker='o', markersize=3)
ax.axhline(80, color='crimson', ls='--', lw=0.8)
ax.axhline(90, color='darkorange', ls='--', lw=0.8)
ax.set_xlabel('Number of Countries (sorted by net revenue)')
ax.set_ylabel('Cumulative Net Revenue Share (%)')
ax.set_title('Geographic Revenue Concentration', fontsize=13, fontweight='bold')
ax.set_ylim(0, 101)
for i in range(min(5, len(ct_sorted))):
    ax.annotate(ct_sorted.iloc[i]['Country'],
                xy=(i+1, ct_sorted.iloc[i]['cum_net_pct']),
                xytext=(i+1.3, ct_sorted.iloc[i]['cum_net_pct']-4),
                fontsize=7, color='navy')
plt.tight_layout()
save_fig('40_geographic_concentration_curve.png')

In [ ]:
# Fig 41 — Country economics: AOV vs revenue per customer (bubble = orders)
fig, ax = plt.subplots(figsize=(10, 7))
for _, row in country_eco.iterrows():
    ax.scatter(row['gross_aov'], row['revenue_per_customer'],
               s=row['orders']/5, alpha=0.6, color='steelblue', edgecolors='navy', lw=0.5)
    ax.annotate(row['Country'], (row['gross_aov'], row['revenue_per_customer']),
                fontsize=7, ha='left', va='bottom')
ax.set_xlabel('Gross AOV (£)')
ax.set_ylabel('Revenue per Customer (£)')
ax.set_title('Country Economics: AOV vs Revenue per Customer\n(bubble size = orders)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
save_fig('41_country_economics_scatter.png')

In [ ]:
# Fig 42 — Product x Country heatmap (top 15 products x top 10 countries)
top15_skus     = pa.head(15)['StockCode'].tolist()
top10_countries= ct_sorted.head(10)['Country'].tolist()
heatmap_data   = (
    prod_country[
        prod_country['StockCode'].isin(top15_skus) &
        prod_country['Country'].isin(top10_countries)
    ]
    .pivot_table(index='canonical_description', columns='Country',
                 values='net_revenue', aggfunc='sum', fill_value=0)
    / 1000
)
fig, ax = plt.subplots(figsize=(13, 8))
sns.heatmap(heatmap_data, ax=ax, cmap='YlOrRd', linewidths=0.3, linecolor='white',
            annot=True, fmt='.0f', annot_kws={'size': 7},
            cbar_kws={'label': 'Net Revenue (£000s)'})
ax.set_title('Product x Country Net Revenue Heatmap\n(Top 15 products x Top 10 countries, £000s)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Country')
ax.set_ylabel('Product')
plt.xticks(rotation=30, ha='right', fontsize=8)
plt.yticks(fontsize=7)
plt.tight_layout()
save_fig('42_product_country_heatmap.png')

---
## 17. Business Questions

Six targeted questions answered from the data.

In [ ]:
top3_net = pa.head(3)['net_revenue'].sum()

bqs = [
    {
        'Q': 'Which products contribute most to net revenue?',
        'Method': 'prod_active sorted by net_revenue.',
        'Finding': f"Top product: {pa.iloc[0]['canonical_description']} (£{pa.iloc[0]['net_revenue']:,.0f} net). "
                   f"Top 3 together: £{top3_net:,.0f}.",
    },
    {
        'Q': 'How concentrated is product revenue?',
        'Method': 'Cumulative net revenue share by product.',
        'Finding': f"{n50} products ({p50:.1f}%) account for 50% of net revenue. "
                   f"{n80} products ({p80:.1f}%) account for 80%.",
    },
    {
        'Q': 'Are high-volume products also high-revenue products?',
        'Method': 'Pearson correlation between units and net revenue; quadrant analysis.',
        'Finding': f"Pearson r = {corr_uv:.3f}. Moderate positive correlation. "
                   f"{len(hv_hq):,} products are simultaneously high-volume and high-price.",
    },
    {
        'Q': 'Which countries contribute most revenue?',
        'Method': 'country_tbl sorted by net_revenue.',
        'Finding': f"Top country: {country_tbl.iloc[0]['Country']} ({country_tbl.iloc[0]['net_rev_share']:.1f}%). "
                   f"Top 3 countries: {s3c:.1f}%. Total markets: {len(country_tbl)}.",
    },
    {
        'Q': 'Do international markets have different customer economics than the UK?',
        'Method': 'UK vs International revenue_per_customer and gross_aov.',
        'Finding': f"UK revenue/customer: £{uk_rpc:,.0f}. International: £{int_rpc:,.0f}. "
                   f"International gross AOV (£{int_aov:,.0f}) is {int_aov/uk_aov:.1f}x the UK (£{uk_aov:,.0f}).",
    },
    {
        'Q': 'Are there products concentrated in one non-UK geography?',
        'Method': 'Products where one country accounts for >= 80% of gross revenue (min £5k gross).',
        'Finding': f"{len(prod_conc_filtered)} products with min £5k gross are dominated by a single non-UK country.",
    },
]

for bq in bqs:
    print(f"Q  : {bq['Q']}")
    print(f"Method  : {bq['Method']}")
    print(f"Finding : {bq['Finding']}")
    print()

---
## 18. Validation

In [ ]:
all_ok = True

def chk(label, cond, detail=''):
    global all_ok
    status = 'PASS' if cond else 'FAIL'
    if not cond:
        all_ok = False
    msg = f'  [{status}] {label}'
    if detail:
        msg += f'  ({detail})'
    print(msg)
    assert cond, f'Validation failed: {label}'

chk('Product gross sum == vmt gross',
    abs(prod_active['gross_revenue'].sum() - vmt['revenue'].sum()) < 0.01)
chk('Product net sum == mpt net',
    abs(prod_tbl['net_revenue'].sum() - mpt['revenue'].sum()) < 0.01)
chk('Country gross sum == vmt gross',
    abs(country_tbl['gross_revenue'].sum() - vmt['revenue'].sum()) < 0.01)
chk('UK + International net == total net',
    abs(geo['net_revenue'].sum() - country_tbl['net_revenue'].sum()) < 0.01)
chk('Product-country net sum == vmt net',
    abs(prod_country['net_revenue'].sum() - vmt['revenue'].sum()) < 0.01)
chk('Return rate denominator > 0 for filtered products',
    (pa_ret['gross_revenue'] > 0).all())
chk('23166 investigation: rows found', len(rows_23166) > 0)
chk('20879 investigation: rows found', len(rows_20879) > 0)
chk('Product concentration n80 < n_active', n80 < n_active)
chk('Geographic concentration top-1 < 100%', s1c < 100)
chk('UK + International orders == vmt orders',
    abs(geo['orders'].sum() - vmt['Invoice'].nunique()) < 1)

print(f'\nAll validation checks: {"PASSED" if all_ok else "FAILED"}')

---
## 19. Save Outputs

In [ ]:
prod_tbl.to_csv(OUTPUT_DIR / 'product_summary.csv', index=False)
country_tbl.to_csv(OUTPUT_DIR / 'country_summary.csv', index=False)
print(f'Saved product_summary.csv  : {len(prod_tbl):,} rows')
print(f'Saved country_summary.csv  : {len(country_tbl):,} rows')

---
## 20. Product & Geographic Intelligence — Findings

All values below are computed from the actual dataset.

### Finding 1: Extreme product revenue concentration

**Finding:** 284 products (5.9% of the 4,837-product catalogue) account for 50% of net revenue. 1,033 products (21.4%) account for 80%.  
**Evidence:** Pareto analysis on `prod_active` sorted by net revenue. n50=284, n80=1033, n_active=4837.  
**Business relevance:** The business is heavily dependent on a small product core. Loss of availability or supply chain disruption for top products would have disproportionate revenue impact. The long tail (3,804 products, 78.6% of catalogue) contributes only 20% of revenue.

---

### Finding 2: Top product — REGENCY CAKESTAND 3 TIER

**Finding:** StockCode 22423 (REGENCY CAKESTAND 3 TIER) is the #1 product by net revenue at £327,345. It has 3,918 orders across 1,314 customers — the highest revenue-per-customer among high-reach products at £261.85.  
**Evidence:** `prod_active` ranked by `net_revenue`. Gross £344,069, return rate 4.9%.  
**Business relevance:** This is both a volume product (3,918 orders) and a high-value product (£261.85 revenue per customer). It warrants dedicated availability management and cross-sell association analysis.

---

### Finding 3: Volume and price are moderately correlated with revenue

**Finding:** Pearson correlation between positive units and net revenue = 0.661. Correlation between average selling price and net revenue = 0.030 (near-zero). 810 products are simultaneously above-median volume and above-median price.  
**Evidence:** `prod_active['positive_units'].corr(prod_active['net_revenue'])`.  
**Business relevance:** Volume drives revenue more than price for most products. High-price/low-volume niche products (1,599) and high-volume/low-price commodities (1,598) each represent distinct margin strategies.

---

### Finding 4: Return rates are generally low but with significant outliers

**Finding:** Among products with >= 10 orders, the median return rate is 0.60%. 95% of products have a return rate below 11.2%. However, 20879 (TREE OF NOAH FESTIVE SCENTED CANDLE) shows 297% (returns exceed sales) and 23166 shows 94.8%.  
**Evidence:** `pa_ret['return_rate_pct'].describe(percentiles=[.5,.95,.99])`.  
**Business relevance:** Both outliers are explained by data-window artefacts (23166: single bulk wholesale cancellation; 20879: returns reference pre-dataset-window transactions). The overall return rate of 3.61% is commercially normal for an e-commerce wholesale operation.

---

### Finding 5: The UK dominates revenue with 85.6% of net revenue from a single market

**Finding:** United Kingdom accounts for £16,350,924 (85.6%) of total net revenue. The top 3 countries account for 91.6%. The business operates across 43 countries.  
**Evidence:** `country_tbl` sorted by `net_revenue`. s1c=85.57%, s3c=91.55%.  
**Business relevance:** Extreme geographic concentration creates risk: any UK market disruption (economic, regulatory, or supply chain) would have a major revenue impact. However, the 14.4% from 42 international markets with higher gross AOV (£853 vs £470) suggests significant international expansion potential.

---

### Finding 6: International customers spend 1.82× more per order than UK customers

**Finding:** International gross AOV = £853.09 vs UK gross AOV = £469.62. International revenue per customer = £5,370 vs UK £3,184.  
**Evidence:** `geo` table grouping by `geo_group`.  
**Business relevance:** International customers are likely wholesale buyers placing larger orders. Their higher AOV and revenue per customer suggest that targeted international expansion (particularly to Netherlands, EIRE, Germany — next three markets) could be disproportionately revenue-accretive relative to the number of customers acquired.

---

### Finding 7: 634 products with min £5k gross are dominated by a single non-UK geography

**Finding:** 634 products where one non-UK country accounts for >= 80% of gross revenue (minimum £5,000 gross threshold). 20 additional products are UK-dominated.  
**Evidence:** `prod_conc_filtered` derived from `prod_country['country_gross_share'] >= 80`.  
**Business relevance:** These geographically-concentrated products may be regional tastes, seasonal gifts relevant to specific markets, or distribution agreements. They represent both geographic risk (single-market dependency) and opportunity (concentrated demand indicates market-specific appeal).

---

### Next Steps

1. **NB06 — Customer Intelligence:** Customer lifetime value, purchase frequency, order value distribution, behavioural segmentation at customer level (identified customers only).
2. **NB07 — RFM Analysis:** Recency, Frequency, Monetary scoring using snapshot date 2011-12-10.
3. **NB08 — Customer Segmentation:** K-means clustering on RFM scores.
4. **NB09 — Cohort & Retention Analysis:** Monthly cohort retention matrices.
5. **NB10 — Market Basket Analysis:** Association rules on positive genuine merchandise baskets.